# Exposure to soybean and palm oil prices from crop suitability (GAEZ)

**Goal.** Build, for each of our 62 countries, a fixed measure of how exposed its forest is to
soybean and oil palm, that does **not** depend on past deforestation. It replaces (or checks) the
current exposure based on 1996-2000 production.

**Idea.** A price boom for a crop should push deforestation where the land *under the forest*
is suitable for that crop. So for each country $i$ and crop $c$:

$$E_{ic} = \frac{\sum_{k \in i} \tfrac{SI_{ck}}{10\,000} \times F_k}{\sum_{k \in i} F_k}$$

- $k$: cells of the 5 arc-minute grid (about 9 × 9 km at the equator) inside country $i$
- $SI_{ck}$: GAEZ v4 suitability index of crop $c$ in cell $k$ (0 to 10 000; high inputs, rainfed, climate 1981-2010)
- $F_k$: tree cover in 2000 in cell $k$ (km² of Hansen pixels with canopy > 30%, the threshold of our outcome)

$E_{ic}$ is the **average suitability of the forest of 2000**. It only depends on soil, terrain and
climate, and on forest measured before our sample (2001-2025).

**Variant.** The same average weighted by land area instead of forest (suitability of the whole country).

**Data** (all in the repository, see `docs/data_log.md`):
- `raw/gaez/sxHr_soy.tif`, `raw/gaez/sxHr_olp.tif`: GAEZ v4 suitability rasters
- `processed/forest2000_5arcmin.csv`: Hansen tree cover 2000 on the same grid, built by `src/build_forest2000_grid.py`
- `raw/boundaries/ne_10m_admin_0_countries.zip`: Natural Earth country borders

**Output:** `processed/exposure_gaez.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.transform import from_origin

# Data is loaded from GitHub with URL links, so the notebook runs on any computer
BASE_URL = "https://raw.githubusercontent.com/AndrewEberhardt/DataScienceProject/exposure/data/"
FIGURES = "../../figures/"
OUTPUT = "../../data/processed/exposure_gaez.csv"

CROPS = {"soy": "Soybeans", "olp": "Palm oil"}   # GAEZ code -> name used in the project

# 5 arc-minute grid of GAEZ: 2160 rows x 4320 columns, row 0 at 90N, column 0 at 180W
CELLS_PER_DEG = 12
N_ROWS, N_COLS = 180 * CELLS_PER_DEG, 360 * CELLS_PER_DEG
GRID = from_origin(-180, 90, 1 / CELLS_PER_DEG, 1 / CELLS_PER_DEG)

SOURCE = "Sources: FAO/IIASA GAEZ v4; Hansen et al. (2013) GFC-2024-v1.12; Natural Earth"
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8984"


def add_source(fig, text=SOURCE):
    # Write the data source under the chart
    fig.text(0.01, 0.005, text, fontsize=8, color="#666666", ha="left", va="bottom")

## 1. Load the data

In [ ]:
sample = pd.read_csv(BASE_URL + "processed/shock_panel.csv")
countries = (sample[["iso3", "country", "region", "extent_2000_ha"]]
             .drop_duplicates("iso3").sort_values("iso3").reset_index(drop=True))
print(len(countries), "countries in the sample")

forest = pd.read_csv(BASE_URL + "processed/forest2000_5arcmin.csv")
print(f"{len(forest):,} grid cells with forest, {forest.forest2000_km2.sum():,.0f} km2")
forest.head()

In [ ]:
# GAEZ suitability rasters: index 0-10000, -9 = no data (water, outside the land mask)
suit = {}
for crop in CROPS:
    with rasterio.open(BASE_URL + f"raw/gaez/sxHr_{crop}.tif") as src:
        assert src.shape == (N_ROWS, N_COLS) and src.transform.almost_equals(GRID)
        a = src.read(1).astype(float)
        a[a == src.nodata] = np.nan
        suit[crop] = a / 10000          # 0 = not suitable, 1 = very high suitability
    print(f"{CROPS[crop]:9s}: suitable (index > 0) on {np.mean(suit[crop][~np.isnan(suit[crop])] > 0):.1%} of land cells")

## 2. Put every grid cell in a country

We draw the Natural Earth borders on the 5' grid. A cell goes to the country that contains its centre.
Coastal cells whose centre falls in the sea would be lost (this matters for small islands such as
Vanuatu or the Solomon Islands), so these cells go to a country that touches them.

In [ ]:
world = gpd.read_file(BASE_URL + "raw/boundaries/ne_10m_admin_0_countries.zip")
world = world[world["ADM0_A3"] != "ATA"].reset_index(drop=True)   # Antarctica, far from our sample
assert set(countries.iso3) <= set(world.ADM0_A3)

shapes = [(geom, k + 1) for k, geom in enumerate(world.geometry)]   # 0 = no country
centre = features.rasterize(shapes, out_shape=(N_ROWS, N_COLS), transform=GRID, fill=0, dtype="int32")
touched = features.rasterize(shapes, out_shape=(N_ROWS, N_COLS), transform=GRID, fill=0, dtype="int32",
                             all_touched=True)
country_id = np.where(centre > 0, centre, touched)

iso_of_id = np.r_[[None], world.ADM0_A3.values]
forest["iso3"] = iso_of_id[country_id[forest.row, forest.col]]
print(f"forest outside any country: {forest.loc[forest.iso3.isna(), 'forest2000_km2'].sum():,.0f} km2")

**Check.** Summed by country, our grid should give back the tree cover in 2000 of Global Forest Watch
(30% threshold), which is the denominator of our outcome.

In [ ]:
check = countries.set_index("iso3")[["country", "extent_2000_ha"]].copy()
check["grid_ha"] = forest.groupby("iso3").forest2000_km2.sum() * 100
check["ratio"] = check.grid_ha / check.extent_2000_ha
print(f"total: grid / GFW = {check.grid_ha.sum() / check.extent_2000_ha.sum():.3f}")
print(f"by country: median {check.ratio.median():.3f}, min {check.ratio.min():.3f}, max {check.ratio.max():.3f}")
check.sort_values("ratio").iloc[np.r_[0:5, -5:0]].round(3)

## 3. Exposure of each country

For each country and crop: the average suitability of its forest (main measure) and of its land (variant).
Cells where GAEZ has no data are left out of both the numerator and the denominator.

In [ ]:
# area (km2) of each row of 5' cells: smaller towards the poles
R = 6371.0072
lat_top = 90 - np.arange(N_ROWS) / CELLS_PER_DEG
cell_km2 = R**2 * np.deg2rad(1 / CELLS_PER_DEG) * (np.sin(np.deg2rad(lat_top)) - np.sin(np.deg2rad(lat_top - 1 / CELLS_PER_DEG)))

# all land cells of the sample countries (for the variant), centre rule as for the forest
land = pd.DataFrame(np.argwhere(centre > 0), columns=["row", "col"])
land["iso3"] = iso_of_id[centre[land.row, land.col]]
land = land[land.iso3.isin(countries.iso3)].reset_index(drop=True)
land["area_km2"] = cell_km2[land.row]

cells_f = forest[forest.iso3.isin(countries.iso3)].copy()
for crop in CROPS:
    cells_f[crop] = suit[crop][cells_f.row, cells_f.col]
    land[crop] = suit[crop][land.row, land.col]


def weighted_mean(df, value, weight):
    ok = df[value].notna()
    return (df.loc[ok, value] * df.loc[ok, weight]).groupby(df.loc[ok, "iso3"]).sum() / df.loc[ok, weight].groupby(df.loc[ok, "iso3"]).sum()


exposure = countries[["iso3", "country", "region"]].set_index("iso3")
for crop, name in CROPS.items():
    key = name.lower().replace(" ", "_")
    exposure[f"suit_forest_{key}"] = weighted_mean(cells_f, crop, "forest2000_km2")
    exposure[f"suit_land_{key}"] = weighted_mean(land, crop, "area_km2")

share_nodata = cells_f.groupby("iso3").apply(lambda d: d.loc[d.soy.isna(), "forest2000_km2"].sum() / d.forest2000_km2.sum())
print(f"forest in cells without GAEZ data: median {share_nodata.median():.2%}, max {share_nodata.max():.1%} ({share_nodata.idxmax()})")
assert exposure.notna().all().all()
exposure.describe().round(3)

**Standardise.** Across the 62 countries, each measure gets mean 0 and standard deviation 1, so a
regression coefficient reads as the effect of one standard deviation more exposure.

In [ ]:
raw_cols = [c for c in exposure.columns if c.startswith("suit_")]
for c in raw_cols:
    exposure["E_" + c.removeprefix("suit_")] = (exposure[c] - exposure[c].mean()) / exposure[c].std()

exposure = exposure.reset_index()
exposure.to_csv(OUTPUT, index=False)
print("saved", OUTPUT, exposure.shape)
exposure.sort_values("suit_forest_soybeans", ascending=False).head(10).round(3)

## 4. Which countries are the most exposed?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 7), sharex=True)
for ax, (crop, name) in zip(axes, CROPS.items()):
    key = name.lower().replace(" ", "_")
    d = exposure.sort_values(f"suit_forest_{key}").tail(20)
    ax.barh(d.country, d[f"suit_forest_{key}"], color=BLUE, height=0.7, label="forest of 2000")
    ax.scatter(d[f"suit_land_{key}"], d.country, color=ORANGE, s=30, zorder=3, label="all land")
    ax.set_title(f"{name}: 20 most exposed forests", loc="left", fontsize=11)
    ax.set_xlabel("average suitability (0 = none, 1 = very high)", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", color="#e5e5e5"); ax.set_axisbelow(True)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, ncol=2, loc="upper left", bbox_to_anchor=(0.01, 0.955))
fig.suptitle("Average GAEZ suitability of the forest of 2000, by country", x=0.01, ha="left", fontsize=13)
fig.tight_layout(rect=(0, 0.03, 1, 0.93))
add_source(fig)
fig.savefig(FIGURES + "exposure_gaez_top20.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
print("Correlation across the 62 countries")
print(f"  forest measure vs land measure, soybeans: {exposure.suit_forest_soybeans.corr(exposure.suit_land_soybeans):.2f}")
print(f"  forest measure vs land measure, palm oil: {exposure.suit_forest_palm_oil.corr(exposure.suit_land_palm_oil):.2f}")
print(f"  soybeans vs palm oil (forest measure):    {exposure.suit_forest_soybeans.corr(exposure.suit_forest_palm_oil):.2f}")

## 5. Comparison with the current exposure (production 1996-2000)

The current price shock uses $\log(1 + E_{ic})$ with $E_{ic}$ the value of 1996-2000 production of crop $c$
per hectare of forest in 2000 (`exposure_weights.csv`). That measure follows *where the crop was already
grown*; the suitability measure follows *where it could grow in the forest*. We compare them with the
rank (Spearman) correlation, since the production measure is very skewed.

In [ ]:
prod = pd.read_csv(BASE_URL + "processed/exposure_weights.csv")
prod["E_prod"] = np.log1p(prod["share"] * prod["value_per_forest_ha"])
prod = prod.pivot(index="iso3", columns="commodity", values="E_prod")

comp = exposure.set_index("iso3")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, name in zip(axes, CROPS.values()):
    key = name.lower().replace(" ", "_")
    x, y = comp[f"suit_forest_{key}"], prod.loc[comp.index, name]
    rho = x.corr(y, method="spearman")
    ax.scatter(x, y, color=BLUE, s=30, alpha=0.8, edgecolor="white", linewidth=0.8)
    for iso in pd.concat([x.nlargest(4), y.nlargest(4)]).index.unique():
        ax.annotate(iso, (x[iso], y[iso]), xytext=(4, 2), textcoords="offset points", fontsize=8, color="#52514e")
    ax.set_title(f"{name} (Spearman correlation {rho:.2f})", loc="left", fontsize=11)
    ax.set_xlabel("suitability of the forest of 2000 (GAEZ)")
    ax.set_ylabel("log(1 + production value per ha of forest), 1996-2000")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Suitability and past production give different exposures", x=0.01, ha="left", fontsize=13)
fig.tight_layout(rect=(0, 0.03, 1, 0.95))
add_source(fig, SOURCE + "; FAOSTAT; World Bank Pink Sheet")
fig.savefig(FIGURES + "exposure_gaez_vs_production.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Summary

*To write once the numbers above are checked.* Points to cover:
- how close the grid is to GFW (section 2)
- which countries are the most exposed, and whether the forest and land measures agree (section 4)
- how different the suitability and production measures are (section 5), and what that means for the regressions

**Next step:** the price shock $X_{it} = \sum_c E_{ic} \times \tilde p_{c,t-1}$ with $E_{ic}$ from
`exposure_gaez.csv`, then the regressions with country and year fixed effects.